当前大语言模型训练通常采用Adam 优化算法，除了需要每个参数梯度之外，还需要一阶动量（Momentum）和二阶动量（Variance）。虽然Adam 优化算法相较SGD 算法通常效果更好也更稳定，但是对计算设备内存的占用显著增大。为了降低内存占用，大多数系统已经采用了混合精度训练（Mixed Precision Training）方式，即同时存在FP16（16 位浮点数）或者BF16（Bfloat16）和FP32（32 位浮点数）两种格式的数值。FP32、FP16 和BF16 表示如图4.15所示。FP32 中第31 位为符号位，第30 到第23 位用于表示指数，第22 到第0 位用于表示尾数。FP16 中第15 位为符号位，第14到第10 位用于表示指数，第9 到第0 位用于表示尾数。BF16 中第15 位为符号位，第14 到第7 位用于表示指数，第6 到第0 位用于表示尾数。由于FP16 的值区间比FP32 的值区间小很多，所以在计算过程中很容易出现上溢出和下溢出。BF16 相较于FP16 以精度换取更大的值区间范围。但是，由于FP16 和BF16 相较FP32 精度低，训练过程中可能会出现梯度消失和模型不稳定的问题。因此，需要使用一些技术来解决这些问题，例如动态损失缩放（Dynamic Loss Scaling）和混合精度优化器（Mixed Precision Optimizer）等。  


混合精度优化的过程如图4.16所示。Adam 优化器状态包括模型参数备份、一阶动量和二阶动量都采用FP32 保存式存储。计算过程中使用的模型参数和梯度采用FP16 格式存储假设。模型参数量为 $\Phi$ ，则共需要 $2\Phi+2\Phi+(4\Phi+4\Phi+4\Phi)=16\Phi$ 字节存储。其中Adam 状态占比 $75\%_{\mathrm{c}}$ 。动态损失缩放反向传播前，将损失变化（dLoss）手动增大 $2^{K}$ 倍，因此反向传播时得到的激活函数梯度则不会溢出；反向传播后，将权重梯度缩小 $2^{K}$ 倍，恢复正常值。举例来说，对于包含75 亿个参数模型，如果用FP16 格式，只需要15GB 计算设备内存，但是在训练阶段模型状态实际上需要耗费120GB。计算卡内存占用中除了模型状态之外，还有剩余状态（Residual States），包括激活值（Activation）、各种临时缓冲区（Buffer）以及无法使用的显存碎片（Fragmentation）等。由于激活值可以用检查点（Activation Checkpointing）方式使得激活值内存占用大幅度减少，因此如何减少模型状态尤其是Adam 优化器状态是解决内存占用问题的关键。 


![](images/319049ce5db4ce9d9bb0750e7e4832484ee565ddeaf4095ec1f2e8821ffbf2b6.jpg)  

![](images/845dd413d9f22e295b03f539e423ec6e592a7c5bb46884bc62deeec3d2954840.jpg) 

# 零冗余优化器（ZeroRedundancyDataParallelism，ZeRO）

目标就是针对模型状态的存储进行去除冗余的优化[139–141]。ZeRO 使用分区的方法，即将模型状态量分割成多个分区，每个计算设备只保存其中的一部分。这样整个训练系统内只需要维护一份模型状态，减少了内存消耗和通信开销。具体来说，如图所示，ZeRO 包含以下三种方法：  

- 对Adam 优化器状态进行分区，图4.17中 $\mathrm{P}_{o s}$ 部分。模型参数和梯度依然是每个计算设备保存一份。此时，每个计算设备所需内存是 $4\Phi+{\textstyle\frac{12\Phi}{N}}$ 字节，其中 $N$ 是计算设备总数。当 $N$ 比较大时，每个计算设备占用内存趋向于 $\mathrm{4\Phi\mathrm{B}}$ ，也就是原来 $16\Phi\mathrm{B}$ 的 $\textstyle{\frac{1}{4}}$ 。  
- 对模型梯度进行分区，图4.17中的 $\mathrm{P}_{o s+g}$ 。模型参数依然是每个计算设备保存一份。此时，每个计算设备所需内存是 $\begin{array}{r}{2\Phi+\frac{2\Phi+12\Phi}{N}}\end{array}$ 字节。当 $N$ 比较大时，每个计算设备占用内存趋向于$2\Phi\mathrm{B}$ ，也就是原来 $16\Phi\mathrm{B}$ 的 $1/8_{\circ}$ 。  
- 对模型参数也进行分区，图4.17中的 ${\mathrm{P}}_{o s+g+p}$ 。此时，每个计算设备所需内存是 $\begin{array}{r}{\frac{16\Phi}{N}\mathbf{B}.}\end{array}$ 。当 $N$ 比较大时，每个计算设备占用内存趋向于0。  


![](images/bc6ccc57f41ca979bf91c76b23274a15f7e50435af31b804a1c954e9f0020835.jpg)  

在DeepSpeed 框架中， $\mathrm{P}_{o s}$ 对应Zero-1， $\mathrm{P}_{o s+g}$ 对应Zero-2， $\mathrm{P}_{o s+g+p}$ 对应Zero-3。文献[141]中也对ZeRO 优化方法所带来的通信量增加情况进行了分析，Zero-1 和Zero-2 对整体通信量没有影响，对通讯有一定延迟影响，但是整体性能影响很小。Zero-3 所需的通信量则是正常通信量的1.5 倍。  

PyTorch 中也实现了ZeRO 优化方法，可以使用ZeroRedundancyOptimizer 调用，也可与“torch.nn.parallel.DistributedDataParallel”结合使用，以减少每个计算设备的内存峰值消耗。使用ZeroRedundancyOptimizer 的参考代码如下所示：

In [ ]:
import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
from torch.distributed.optim import ZeroRedundancyOptimizer
from torch.nn.parallel import DistributedDataParallel as DDP
def print_peak_memory(prefix, device):
    if device == 0:
        print(f"{prefix}: {torch.cuda.max_memory_allocated(device) // 1e6}MB ")
def example(rank, world_size, use_zero):
    torch.manual_seed(0)
    torch.cuda.manual_seed(0)
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '29500'
    # 创建默认进程组
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    # 创建本地模型
    model = nn.Sequential(*[nn.Linear(2000, 2000).to(rank) for _ in range(20)])
    print_peak_memory("Max memory allocated after creating local model", rank)
    # 构建 DDP 模型
    ddp_model = DDP(model, device_ids=[rank])
    print_peak_memory("Max memory allocated after creating DDP", rank)
    # 定义损失函数和优化器
    loss_fn = nn.MSELoss()
    if use_zero:
        optimizer = ZeroRedundancyOptimizer( # 这里使用了 ZeroRedundancyOptimizer
            ddp_model.parameters(),
            optimizer_class=torch.optim.Adam, # 包装了 Adam
            lr=0.01
        )
    else:
        optimizer = torch.optim.Adam(ddp_model.parameters(), lr=0.01)
    # 前向传播
    outputs = ddp_model(torch.randn(20, 2000).to(rank))
    labels = torch.randn(20, 2000).to(rank)
    # 反向传播
    loss_fn(outputs, labels).backward()
    # 更新参数
    print_peak_memory("Max memory allocated before optimizer step()", rank)
    optimizer.step()
    print_peak_memory("Max memory allocated after optimizer step()", rank)
    print(f"params sum is: {sum(model.parameters()).sum()}")
    
def main():
    world_size=2
    print("===Using ZeroRedundancyOptimizer ===")
    mp.spawn(example,
        args=(world_size, True),
        nprocs=world_size,
        join=True)
    print("===Not Using ZeroRedundancyOptimizer===")
    mp.spawn(example,
        args=(world_size, False),
        nprocs=world_size,
        join=True)
    
if __name__=="__main__":
    main()

执行上述代码，可以得到如下输出：
```bash
=== Using ZeroRedundancyOptimizer===
Max memoryallocatedaftercreatinglocalmodel:335.0MB
Max memoryallocatedaftercreatingDDP:656.0MB
Max memoryallocatedbefore optimizerstep():992.0MB
Max memoryallocatedafteroptimizerstep():1361.0MB
paramssumis:-3453.6123046875
paramssumis:-3453.6123046875
=== NotUsing ZeroRedundancyOptimizer===
Max memoryallocatedaftercreatinglocalmodel:335.0MB
Max memoryallocatedaftercreatingDDP:656.0MB
Max memoryallocatedbefore optimizerstep():992.0MB
Max memoryallocatedafteroptimizerstep():1697.0MB
paramssumis:-3453.6123046875
paramssumis:-3453.6123046875
```